# 04 CNN Training

Stage 7 prepares PyTorch-ready tensors for deep learning, Stage 8 trains the first validation-monitored 1D CNN, Stage 9 compares basic training choices for that same single-epoch CNN, and Stage 10 compares simple and temporal-context CNNs on matched context-eligible center epochs. This notebook starts with shape, leakage, and preprocessing checks, then runs guarded validation-only training workflows through reusable `src.train` utilities. The held-out test split is not evaluated here.


This setup cell imports the reusable data and training utilities, resolves paths whether the notebook is run from the repository root or the `notebooks/` directory, and defines a small debug configuration for the initial Stage 8 smoke run. The expected output is a single Boolean indicating whether local raw data and `data/interim/epoch_index.csv` are available. If it is `False`, later cells skip cleanly instead of failing on missing local DREAMT artifacts. Stage 9 and Stage 10 cells remain guarded so routine execution does not launch long training runs.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)
from src.train import (
    DEFAULT_STAGE8_OUTPUT_DIR,
    DEFAULT_STAGE9_OUTPUT_DIR,
    DEFAULT_STAGE10_OUTPUT_DIR,
    TrainConfig,
    build_stage9_screening_configs,
    build_stage10_comparison_configs,
    run_tiny_overfit_test,
    run_stage9_experiments,
    run_stage10_experiments,
    train_model,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3
EPOCHS = 1

raw_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
metadata_path = repo_root / DEFAULT_PREPROCESSING_METADATA_PATH
output_dir = repo_root / DEFAULT_STAGE8_OUTPUT_DIR
stage9_output_dir = repo_root / DEFAULT_STAGE9_OUTPUT_DIR
stage10_output_dir = repo_root / DEFAULT_STAGE10_OUTPUT_DIR
stage8_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=output_dir,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
artifacts_available


## Build Single-Epoch Datasets

This section builds the PyTorch-ready single-epoch datasets used by the first CNN. The first cell fits median imputation and per-channel standardization metadata from training epochs only, then saves that metadata for reuse. Expected output is either no displayed output when local artifacts are present, or a skip message when raw files or the epoch index are absent. Validation and test data are not used to fit preprocessing statistics.

In [ ]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


This cell reloads the saved preprocessing metadata, applies it to train and validation datasets, checks that participant-level split leakage is absent within each dataset, and constructs DataLoaders. The printed batch shape should be `(batch, channels, timepoints)` for `x` and one integer label per epoch for `y`; the participant counts confirm the debug subset size; and the metadata channels should match the configured model channels. The test split is intentionally not loaded.

In [ ]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    loaders = {
        "train": torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        "validation": torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
    }
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants)})
    print("metadata channels:", stats["channels"])


## Temporal Context And Sequence Shape Checks

This section performs lightweight shape checks for the temporal-context and sequence datasets that will support later CNN-context and CNN-GRU experiments. These checks answer whether neighboring-epoch windows can be formed without crossing participant boundaries and whether the tensor shapes match the intended model families. The expected output is one example context item and one example sequence item when enough consecutive training epochs exist in the debug subset.

In [ ]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


## Tiny Overfit Smoke Test

This cell runs a tiny repeated-batch overfit test on training data only. Its purpose is to verify that the model, loss, optimizer, tensor dtypes, and device handling can reduce training loss before running validation-monitored experiments. The expected output is the first and last loss; the last value should usually be lower than the first. This is a plumbing check, not a model-selection result.

In [ ]:
if artifacts_available:
    overfit_history = run_tiny_overfit_test(train_ds, stage8_config)
    print("loss first/last:", round(overfit_history["loss"].iloc[0], 4), round(overfit_history["loss"].iloc[-1], 4))


## Single-Epoch CNN Training

This cell trains the first modest single-epoch CNN and monitors validation metrics after each epoch. In the default debug configuration, it runs for one epoch over a small participant subset, so the goal is confirmation that artifacts are produced rather than strong performance. Expected outputs are the training-history table, the best epoch, and the output directory containing history, validation metrics, confusion matrix, plots, and checkpoints. The validation split is used for monitoring; the test split remains untouched.

In [ ]:
if artifacts_available:
    training_result = train_model(loaders["train"], loaders["validation"], stage8_config)
    display(training_result.history)
    print("best epoch:", training_result.best_epoch)
    print("outputs:", training_result.output_dir)


## Stage 9 Training-Choice Experiments

Stage 9 keeps the model family fixed to the single-epoch CNN and compares basic training choices using validation macro F1 as the primary selection metric. The first cell defines the Stage 9 base configuration and expands it into a controlled screening grid over unweighted versus train-only class-weighted loss, learning rate, dropout including `0.0`, and weight decay. The expected output is the number of configured runs. These configurations still use only the train and validation splits; the test split remains untouched.

In [ ]:
stage9_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage9_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=8,
    train_eval_interval=None,
    max_train_participants=None,
    max_val_participants=None,
)

stage9_screening_configs = build_stage9_screening_configs(
    base_config=stage9_base_config,
    output_dir=stage9_output_dir,
    learning_rates=(3e-4, 1e-3, 3e-3),
    dropouts=(0.0, 0.10, 0.25),
    weight_decays=(0.0, 1e-4, 1e-3),
    class_weighting_options=(False, True),
    batch_sizes=(32,),
)
len(stage9_screening_configs)


The full default grid has 54 runs. For a quick local dry run, slice `stage9_screening_configs` before calling `run_stage9_experiments`; for the main Stage 9 result, run the full list on the local machine/GPU. The next cell is guarded by `RUN_STAGE9_EXPERIMENTS = False` so routine notebook execution does not accidentally launch a long training sweep. When enabled, the expected output is a top-10 validation summary sorted by macro F1 plus a results directory containing per-run histories, validation metrics, confusion matrices, checkpoints, and aggregate Stage 9 summary files.

In [ ]:
RUN_STAGE9_EXPERIMENTS = False

if artifacts_available and RUN_STAGE9_EXPERIMENTS:
    stage9_summary = run_stage9_experiments(
        stage9_screening_configs,
        output_dir=stage9_output_dir,
    )
    display(stage9_summary.sort_values("macro_f1", ascending=False).head(10))
    print("outputs:", stage9_output_dir)
else:
    print("Stage 9 experiments are configured but not run in this notebook execution.")


## Stage 10 Temporal-Context CNN Comparison

Stage 10 asks whether neighboring epochs improve validation performance relative to the simple 1D CNN. The comparison is deliberately conservative: it keeps the CNN architecture and broad training defaults fixed, runs only `context_radius=1` and `context_radius=2`, and pairs each context CNN with a simple CNN trained and evaluated on the same context-eligible center epochs. This means the validation comparison is between center-only input and center-plus-neighbor input, not between different validation epoch sets. The test split remains untouched.


In [ ]:
stage10_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage10_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=8,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.10,
    class_weighting=False,
    train_eval_interval=None,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

stage10_comparison_configs = build_stage10_comparison_configs(
    base_config=stage10_base_config,
    output_dir=stage10_output_dir,
    context_radii=(1, 2),
)
[(config.model_name, config.context_radius, config.comparison_context_radius) for config in stage10_comparison_configs]


The next cell is guarded by `RUN_STAGE10_EXPERIMENTS = False`. When enabled, it trains four validation-only runs: simple CNN and context CNN for radius 1, then simple CNN and context CNN for radius 2. The output summary includes validation macro F1, balanced accuracy, class-level metrics, the paired context radius, and the number of matched center epochs used for each comparison. Broader hyperparameter tuning is intentionally deferred to Stage 11.


In [ ]:
RUN_STAGE10_EXPERIMENTS = False

if artifacts_available and RUN_STAGE10_EXPERIMENTS:
    stage10_summary = run_stage10_experiments(
        stage10_comparison_configs,
        output_dir=stage10_output_dir,
    )
    display(stage10_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage10_output_dir)
else:
    print("Stage 10 experiments are configured but not run in this notebook execution.")
